# Khung báo cáo và bảng giả thuyết — evidence đã có

Thành phần kết quả đã được README dự kiến. Giữ hệ C01–C55; tiến độ cập nhật chính trong docs/experiments.md. Notebook này chỉ chuẩn bị phân tích/biểu đồ từ artifact, không chạy training/evaluation hoặc Git. Tối 13/10 là mục tiêu ưu tiên; hạn toàn bộ tối 14/10/2026, giờ Việt Nam; 15/10 ôn tập, 16/10 bảo vệ.

## Bài toán và phương pháp
Khử nhòe chuyển động không đồng nhất theo không gian. Backbone NAFNet pretrained; bộ ước lượng E dự đoán bản đồ; SFT điều biến feature; A3 bổ sung nhánh channel. Phân biệt A0/SA-10, A0+/SA-11, A1/SA-12, A2/SA-13, A3/SA-14. Data/degradations/models/classical/engine/metrics/utils là hạ tầng được mô tả dùng chung với đồ án CV; modulation là phần nghiên cứu DL. README/CORE_VERSION ghi core-v0.1.0, CHANGELOG có core-v0.1.1; chưa đối chiếu repo CV, không coi khai báo chia sẻ đã hoàn tất.

### Thiết kế đối chứng và công thức huấn luyện

Mục tiêu chính là kiểm tra liệu giám sát bản đồ mức nhòe giúp khử nhòe tốt hơn điều biến học tự do. A0 là backbone NAFNet; A0+ bổ sung hai NAFBlock tại dec1 để kiểm soát phần tăng tham số; A1 và A2 có cùng kiến trúc estimator/SFT, khác trọng số loss bản đồ. A3 bổ sung nhánh điều biến channel, hiện chưa có run full hợp lệ. So sánh A2/A1 dùng ba seed theo kế hoạch; A2/A0+ và các đối chứng A0/A0+ chỉ seed1 trong phạm vi lõi.

Estimator nhận ảnh RGB nhòe và dự đoán Ŝ một kênh ở 1/4 độ phân giải bằng sigmoid. SFT head nội suy Ŝ bilinear tới từng stage và sinh Δγ_s, β_s; với A1/A2, feature sau norm1 được điều biến bằng F̃=(1+Δγ_s)⊙LN(F)+β_s. Các lớp cuối SFT khởi tạo bằng0; nhánh residual khử nhòe dự đoán ảnh nét từ ảnh đầu vào. Loss dùng L_rec=mean(sqrt((Î−I)²+10⁻⁶)) và L_map=SmoothL1(Ŝ,area_resize(S)). Tổng L=L_rec+λ_map L_map; λ_map=0 choA1,0,1 choA2/A3. Gradient restoration truyền về estimator, không detach trong các run lõi; không thêm perceptual/FFT loss để diễn giải các đối chứng này.

Training sử dụng checkpoint pretrained công khai cho mỗi run/seed mới, optimizer/scheduler/RNG mới, patch256,batch8,AdamW lr10⁻⁴,betas(0,9;0,9),weight_decay10⁻³,warmup1000 rồi cosine về10⁻⁷,FP32 vàclip1. Checkpoint đem đánh giá là final của run đủ30.000bước; validation chỉ giám sát và kiểm tra chốt chặn, không chọn checkpoint tốt nhất. Các sai khác data/split bên dưới vẫn phải được giải quyết theo Definition of Done.

## Thiết lập thí nghiệm và sai khác đặc tả
TLC là giao thức chính, FP32, checkpoint final 30.000 bước; validation gốc 40 ảnh chỉ theo dõi và diagnostic, không thay thế test. Training 40k có thể điều chỉnh sau pilot: QĐ-04 đã chốt 30k. Các sai khác khác chưa đủ bằng chứng chấp thuận Definition of Done gốc.

| Hạng mục | Đặc tả gốc do người dùng xác nhận | Giao thức/config hiện tại | Bằng chứng và giới hạn |
|---|---:|---:|---|
| Validation | 400 | 40 ảnh đã xác minh | Config 10/R; CHANGELOG ghi 50/R = 200, chưa thống nhất |
| Test in | 1.200 | 400 theo config | Chưa có testset/hash đăng ký được thu local |
| OOD | 300 | 100 theo config | Chưa có artifact test được xác minh |
| Kernel thẳng / cong | 150 mỗi split | 50 mỗi split theo config | Chưa có artifact test được xác minh |
| Training | 40k, điều chỉnh theo pilot | 30k | QĐ-04 có quyết định; bốn seed 1 đã xác minh |

H-bin vẫn cần evidence EDA. Không tự thay đổi config/manifests hoặc trình bày các quy mô nhỏ hơn là hoàn thành đặc tả gốc.

## Bảng giả thuyết — khung, chưa phải kết luận
| Phép thử | Evidence cần | Phạm vi seed | Trạng thái kết luận |
|---|---|---|---|
| A1 so A0: điều biến | GoPro seed1 đã VerifiedLocal; float PSNR theo video | So sánh seed1; không suy ra ba seed cho A0 | GoPro: +0,0015343 dB, p=0,625; chưa có bằng chứng cải thiện, các split khác pending |
| A2 so A1: giám sát bản đồ, phép thử chính | GoPro seed1 có; còn cặp seed2/3 và test tổng hợp | Ba seed A2 và A1 theo kế hoạch | Seed1: -0,0236103 dB theo video, p=0,048828125 chưa hiệu chỉnh; chưa kết luận nhiều seed |
| A2 so A0+: kiểm soát tham số | GoPro seed1 và tham số có; latency/GMACs còn thiếu | Một seed/model trong phạm vi lõi | Seed1: -0,0193319 dB theo video, p=0,048828125 chưa hiệu chỉnh; các split khác pending |
| A3 so A2: nhánh channel | Full training/evaluation/chi phí còn thiếu | Seed1 | Pilot và diagnostic A3 abort tại counter84; V1/357568385 VerifiedLocal evidence: first nonfinite forward35, fulltraining chưa đạt |
| Oracle/shuffle: vai trò bản đồ | Checkpoint thật và test tổng hợp cố định | Nêu rõ seed checkpoint | Chưa có CSV |

Không bổ sung ngưỡng thắng/p-value sau khi xem số. Chi tiết ngưỡng DoD cần đối chiếu đặc tả gốc ngoài repo.

## Evidence hiện có và cách trình bày
Bốn checkpoint seed 1 đã VerifiedLocal trong phạm vi checkpoint, không phải Accepted. GoPro SA-13 recovery: xunanhv/sad-eval-sa13-recovery V1/scriptVersionId 357357176, đủ 1.111 ID, TLC/FP32; CSV local results/eval/SA-13_eval_gopro_test_pred_tlc.csv, SHA-256 3ce67f7d59a91d65dfb3f7b2b655c291691056d174b9b36902b8e94637795a1e. Giá trị công bố lấy từ CSV/metadata đã xác minh, không chép lại số đã làm tròn để tính bảng.

PSNR uint8 SA-13 khoảng 28,3436 dB, thấp hơn SA-00 khoảng 4,5269 dB: phát hiện cần giải thích, chưa chứng minh lỗi implementation hay SFT thất bại. SSIM SA-00 khác implementation/phạm vi mẫu, không so trực tiếp. Controls chưa có Saved Version đã xác minh tại thời điểm tiếp quản 00:09 ngày 12/10; cập nhật khi có evidence mới, không đặt số giả.

## Kết luận estimator ở đúng phạm vi
Diagnostic đã chạy trên 40 validation NPZ gốc V2/355813018, SA-13_seed1/final.pth; 30 ảnh R ≥ 2. MAE model 0,1769647143 so với constant_val_mean 0,1773214638; baseline hằng dùng trung bình ground truth trên chính validation, chỉ là đối chiếu diagnostic. Spearman trong ảnh R ≥ 2 = 0,0476140375; pooled pixels = 0,0362284104. Mean Ŝ qua ảnh nằm 0,369620–0,377562 trong khi mean S nằm 0,003943–0,690115; tỷ số độ lệch chuẩn trong ảnh khoảng 0,058103. Calibration gần phẳng quanh Ŝ ≈ 0,373.

Kết luận mô tả: checkpoint A2 seed 1 chưa thể hiện bản đồ có biến thiên và tương quan tốt trên validation này. Không kết luận nguyên nhân, dữ liệu không học được hoặc SFT không có ích. Probe E độc lập chưa chạy. Các kiểm tra toy trong results/audit/2026-10-11/mechanism-probe.json chỉ xác minh đường gradient/nhãn trên mô hình nhỏ; không thay thế kiểm tra checkpoint thật. Heading kết luận trong docs/experiments.md phải được người dùng commit; analysis/eval_tables.py còn yêu cầu file đó không có sửa chưa commit. Notebook không bỏ qua gate.

## Hình ảnh, chi phí và trường hợp thất bại
Đường cong HTML hiện chỉ SA-10/13 từ giai đoạn pair-2 còn blind; gate đã mở tại commit a8cb0b2. Dùng step làm trục X, validation PSNR ghi rõ synthetic val 40; không so total loss khi lambda_map khác. HTML đã có tại results/eval/training_curves_seed1_blind.html; không tạo lại/ghi đè.

Ảnh định tính cần prediction thật, cùng ID và crop cho các model, có blur/GT/pred/Ŝ/S nếu split có ground-truth map. Hình generator trong docs/figures chỉ minh họa cơ chế dữ liệu, không phải qualitative restoration. Có trường hợp tốt/trung bình/thất bại và ghi rõ cách chọn; chưa có ảnh prediction TLC được thu. Tham số/GMACs/tốc độ/VRAM phải ghi kích thước ảnh, GPU, batch, precision, TLC và warmup, không suy từ thời gian validation.

### Bộ ID định tính chuẩn bị trước khi đọc controls

Tám ID dưới đây chọn bằng quy tắc không dùng PSNR: với mỗi nhóm video_in_train = 0/1, lấy bốn video đầu theo thứ tự tên và frame ở giữa danh sách ID đã sắp xếp của video đó. Có bốn video chưa xuất hiện trong train và bốn video có phần trùng. Đây là tập minh họa nhỏ, không phải mẫu ngẫu nhiên hoặc bằng chứng thắng toàn test. Giữ cùng ID cho mọi model; không tạo visual_set.txt/manifests mới.

Crop chung dự kiến: x = 512, y = 232, width = height = 256 trên ảnh gốc 1280 x 720, tọa độ pixel từ góc trên trái. Chưa có prediction TLC hoặc crop xuất ra.

| ID GoPro | video_in_train |
|---|---:|
| test/GOPR0396_11_00/blur/000051.png | 0 |
| test/GOPR0410_11_00/blur/000168.png | 0 |
| test/GOPR0854_11_00/blur/000051.png | 0 |
| test/GOPR0862_11_00/blur/000039.png | 0 |
| test/GOPR0384_11_05/blur/004001.png | 1 |
| test/GOPR0385_11_01/blur/003061.png | 1 |
| test/GOPR0868_11_00/blur/000051.png | 1 |
| test/GOPR0871_11_00/blur/000051.png | 1 |

Khảo sát thất bại riêng SA-13: năm ảnh PSNR float thấp nhất của CSV verified là GOPR0854_11_00/000017, 000016, GOPR0862_11_00/000069, GOPR0869_11_00/000044 và GOPR0854_11_00/000018. Đây là lựa chọn theo kết quả SA-13, phải gắn nhãn exploratory và không dùng làm tập so sánh đại diện. Chưa diễn giải nguyên nhân khi chưa xem blur/GT/pred.

### Snapshot BLIND trước mở gate và giới hạn chi phí

Tại snapshot trước commit a8cb0b2, analysis/eval_tables.py đã xuất results/eval/tables/summary.csv, comparisons.json và README.txt ở chế độ BLIND. Summary chỉ có SA-13 GoPro; comparisons rỗng. Các ô map/local-high là thiếu metric trên GoPro, không phải lỗi metrics PSNR/SSIM. Đây là mô tả lịch sử; bảng hiện hành sau gate nằm ở mục GoPro UNBLIND bên dưới.

benchmark_efficiency.py hiện dựng model mặc định và gọi latency() với amp=True, pooling global; chưa có lựa chọn TLC/FP32 trong CLI. Vì vậy không thể dùng output mặc định để mô tả chi phí giao thức bảng chính. Đề xuất bổ sung config/pooling/precision rõ ràng và metadata, giữ benchmark 50 warmup + 200 lượt, batch1, không decode/I/O; chưa sửa hoặc chạy. fvcore nếu thiếu trả MACs=None, phải báo thiếu thay vì 0.


Bảng tham số đã đếm lúc khoảng 00:34 ngày 12/10 bằng load_config(), build_variant() và metrics.efficiency.count_params() trên CPU. Không load checkpoint, không forward/inference hoặc training. Đây là số theo kiến trúc/config hiện tại, không xác nhận training A3 đã hoàn tất.

| Model / variant | Số tham số | Triệu tham số | Phần tăng so A0 |
|---|---:|---:|---:|
| SA-10 / A0 | 17.111.907 | 17,111907 | 0 |
| SA-11 / A0+ | 17.349.731 | 17,349731 | 237.824 |
| SA-12 / A1 | 17.342.180 | 17,342180 | 230.273 |
| SA-13 / A2 | 17.342.180 | 17,342180 | 230.273 |
| SA-14 / A3 | 17.672.660 | 17,672660 | 560.753 |

A0+ thêm hai block ở dec1, lệch phần tăng A2 khoảng3,2792%, phù hợp QĐ-06. GMACs/latency/VRAM benchmark TLC/FP32 vẫn chưa đo; không lấy thời gian evaluation có decoding/I/O làm latency thuần model.

Khung slide vẫn là dàn ý ở notebook này. Runtime load_workspace_dependencies và artifact-tool của skill Presentations không khả dụng trong phiên; chưa tạo PPTX hoặc tuyên bố đã kiểm tra layout.

## Khung slide đồng bộ với báo cáo
1. Bài toán, mục tiêu và phạm vi đã chốt.
2. Dữ liệu, cặp blur/sharp và sơ đồ NAFNet + E + SFT.
3. Giả thuyết/đối chứng và phạm vi seed.
4. Giao thức, các quyết định đã khóa và sai khác đặc tả.
5. Bảng kết quả từ CSV verified; ô thiếu ghi chưa có evidence.
6. Diagnostic estimator; tách quan sát và giả thuyết nguyên nhân.
7. Hình định tính/crop và trường hợp thất bại khi có prediction thật.
8. Hiệu năng, giới hạn, đóng góp DL/phần dùng chung CV.
9. Kết luận theo bằng chứng, hướng phát triển và demo/tái lập.

Đây là dàn ý, chưa phải slide deck hoàn chỉnh.

## Tái lập, demo và bàn giao
Gói A2 cần checkpoint/config/hash, nguồn pretrained/commit, giao thức, evidence, license và hướng dẫn. Người dùng quyết định phát hành/Git; chưa tự publish. Backup độc lập cần kiểm tra hash. Chưa Accepted toàn dự án; hạn tối 14/10 còn phụ thuộc quota, pilot và các đánh giá.


### GoPro seed 1 — bảng UNBLIND đã kiểm chứng (12/10/2026)

Gate được công cụ xác nhận tại commit a8cb0b2, 2026-10-12T11:31:26+07:00. Cả bốn CSV đã VerifiedLocal, cùng 1111 ID, TLC/FP32 và evaluator/src/manifests. Toàn bộ mean và bốn phép so sánh đã được đối chiếu lại bằng CSV và hàm thống kê hiện có, không chạy model mới.

| Model / run | PSNR float (dB) | PSNR uint8 (dB) | SSIM skimage | Ảnh |
|---|---:|---:|---:|---:|
| A0 / SA-10_seed1 | 28.3699 | 28.3649 | 0.8717 | 1111 |
| A0+ / SA-11_seed1 | 28.3671 | 28.3622 | 0.8717 | 1111 |
| A1 / SA-12_seed1 | 28.3700 | 28.3651 | 0.8717 | 1111 |
| A2 / SA-13_seed1 | 28.3485 | 28.3436 | 0.8713 | 1111 |

Summary gộp theo ảnh; comparisons gộp PSNR float theo 10 source-video bằng implementation video_of(), cho mỗi video cùng trọng số. Vì thế diff comparison không nhất thiết bằng phép trừ hai mean ảnh. Phép thử chính A2-A1: -0.0236103 dB, bootstrap 95% CI [-0.0575773,-0.0017533], Wilcoxon p=0.048828125 chưa hiệu chỉnh; A2-A0+: -0.0193319 dB, p=0.048828125; A1-A0: +0.0015343 dB, p=0.625; A2-A0: -0.0220760 dB, p=0.048828125. Tất cả mới seed 1; hiệu ứng nhỏ, p của ba phép gần 0.05, chưa kết luận ba seed hoặc chấp nhận toàn dự án. Giữ nguyên phép kiểm/ngưỡng đã khai báo, không sửa phương pháp để thay nhãn theo kết quả.

Các nan ở psnr_local_high/map_mae/map_spearman là metric không có ground-truth smap/vùng nhòe trên GoPro; core PSNR/uint8/SSIM đều hữu hạn. H groups vẫn bị chặn bởi data.h_bins chưa có; không tạo H từ CSV GoPro. So với tham chiếu SA-00 TLC uint8 ~32.870 dB trong docs, cả bốn model fine-tune thấp khoảng 4.5 dB, không chỉ A2. Đây là phát hiện cần rà phần chung fine-tune/data/protocol, chưa chứng minh lỗi implementation, quên kiến thức hay SFT thất bại. SSIM SA-00 tập con223 khác giao thức, không so trực tiếp.

Bằng chứng: results/eval/tables/summary.csv SHA-256 4ade4d519c33af7f8f9f28af03de166df4404b0b46640b36d1608569bd0777ab; comparisons.json SHA-256 f4834498fc6a2e3f9a6f5d14f47c407deb15b0995e66830d71145068c407403a; README.txt SHA-256 934823349c1d125bc900c628bcd74e15d00bc9b588f584132515eb4c697fd334. Source receipts: results/artifacts/kaggle/357428270/download-provenance.json và 357357176/download-provenance.json.

### Diễn giải kết quả sơ bộ và giới hạn suy luận

Training seed1 SA10/SA13 có300điểm log và13lần validation mỗi run. Charbonnier cuối lần lượt0,009137704/0,009130239; validation synthetic40 cuối33,791469/33,803355dB. Hai đường gần nhau và tiếp tục có gradient tại các điểm log, phù hợp với training hoạt động; không suy ra mọi optimizer-step đều đã kiểm hoặc ưu thế test từ chênh validation0,011887dB. Learning rate cuối10⁻⁷ phù hợp schedule đã khóa.

Trên GoPro seed1, giám sát bản đồ chưa mang lại lợi ích soA1; effect nhỏ âm và mới một seed. Diagnostic validation củaA2 cho bản đồ gần hằng vàrho≈0,0476; hai quan sát phù hợp với cơ chế điều biến phụ thuộc bản đồ chưa tạo ích lợi rõ trong setup hiện tại. Đây là mô tả, chưa phải bằng chứng nguyên nhân: probeE độc lập,oracle/shuffle và kết quả test seed2/3 vẫn thiếu.

Cả bốn model fine-tune cóGoPro PSNR uint8 quanh28,34–28,37dB, thấp khoảng4,5dB soSA00 pretrained công khai. VìA0/A0+/A1 cũng giảm, cần rà phần chung fine-tune, dữ liệu và giao thức trước khi quy mức giảm cho giám sát bản đồ. Chưa có kiểm tra can thiệp để kết luận lỗi implementation, forgetting hoặc nguyên nhân duy nhất. Validation synthetic vàGoPro real blur khác phân phối, không dùng chênh hai tập để tự kết luận overfit. SSIMSA00 khácimplementation/scope và không dùng làm so sánh trực tiếp.

SA14 chưa có kết quả full; diagnostic chỉ định vị output nonfinite đầu tiên được quan sát và việc optimizer làm hỏng tham số sau đó. Không dùng run abort hoặc checkpoint thiếu bước trong bảng model đầy đủ. Sáu trênchín lượt training lõi đã hoàn tất, nhưng tỷ lệ này không phải phần trăm hoàn thành toàn dự án. LPIPS,synthetic/OOD/robustness/RealBlur,chi phí,ảnh định tính và các phần bàn giao còn phải đạt tiêu chí riêng.


In [ ]:

import hashlib,json
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
root=Path.cwd().resolve()
if not (root/'AGENTS.md').exists() and (root.parent/'AGENTS.md').exists():root=root.parent
assert (root/'AGENTS.md').exists(),'Run from repo root or notebooks/'
runs={'SA-10_seed1':root/'results/artifacts/kaggle/355813018/original/sad/results/SA-10_seed1',
 'SA-13_seed1':root/'results/artifacts/kaggle/355813018/original/sad/results/SA-13_seed1'}
fig=make_subplots(rows=3,cols=2,subplot_titles=('Charbonnier training','Validation PSNR (monitoring, not GoPro test)',
 'Map loss (SA-13)','Validation map Spearman (R >= 2, SA-13)','Learning rate','Gradient norm'))
colors={'SA-10_seed1':'#2878B5','SA-13_seed1':'#E07A25'};provenance=[]
for run,folder in runs.items():
 train=[json.loads(s) for s in (folder/'curves.csv').read_text().splitlines() if s.strip()]
 val=[json.loads(s) for s in (folder/'val.jsonl').read_text().splitlines() if s.strip()]
 assert len(train)==300 and train[-1]['step']==30000 and len(val)==13
 for filename in ('curves.csv','val.jsonl'):
  p=folder/filename;provenance.append(f'{p.relative_to(root).as_posix()} SHA-256={hashlib.sha256(p.read_bytes()).hexdigest()}')
 def add(rows,key,row,col,label=run):
  fig.add_trace(go.Scatter(x=[r['step'] for r in rows],y=[r[key] for r in rows],mode='lines',
   name=label,legendgroup=run,showlegend=(row==1 and col==1),line={'color':colors[run]}),row=row,col=col)
 add(train,'charbonnier',1,1);add(val,'val_psnr',1,2);add(train,'lr',3,1);add(train,'grad_norm',3,2)
 if run=='SA-13_seed1':add(train,'l_map',2,1);add(val,'val_spearman',2,2)
fig.update_xaxes(title_text='Optimizer step');fig.update_yaxes(title_text='PSNR (dB)',row=1,col=2)
fig.update_yaxes(title_text='Spearman rho',row=2,col=2);fig.update_yaxes(type='log',row=3,col=1)
fig.update_layout(height=1050,width=1350,template='plotly_white',title='Seed1 training evidence ? SA-10 / SA-13 only (unblind gate retained)')
html=fig.to_html(full_html=True,include_plotlyjs=True,config={'toImageButtonOptions':{'format':'png','scale':2}})
from html import escape
notes='<p>Validation monitors the original40 synthetic images; it is not GoPro test evaluation. SA-11/12 remain hidden pending the committed estimator conclusion. Total losses are not compared because lambda_map differs.</p><pre>'+escape(chr(10).join(provenance))+'</pre>'
html=html.replace('</body>',notes+'</body>');output=root/'results/eval/training_curves_seed1_blind.html'
assert not output.exists(),'Existing chart kept; do not overwrite.'
with output.open('x',encoding='utf-8') as f:f.write(html)
print(output)


In [ ]:
import hashlib, json, math
from pathlib import Path
from html import escape
import plotly.graph_objects as go
from plotly.subplots import make_subplots
root = Path.cwd().resolve()
if not (root / 'AGENTS.md').exists() and (root.parent / 'AGENTS.md').exists():
    root = root.parent
assert (root / 'AGENTS.md').is_file()
source = root / 'results/artifacts/kaggle/355813018/estimator_diag.json'
raw = source.read_bytes()
record = json.loads(raw)
s = record['summary']
assert s['images'] == len(record['per_image']) == 40 and s['images_R_ge_2'] == 30
bins = s['calibration']
for b in bins:
    assert all(math.isfinite(float(b[k])) for k in ('S_mean', 'S_hat_mean', 'S_hat_std'))
fig = make_subplots(rows=1, cols=2, subplot_titles=('Calibration on original validation (40 images)', 'Map MAE: diagnostic constant uses validation target mean'))
fig.add_trace(go.Scatter(x=[b['S_mean'] for b in bins], y=[b['S_hat_mean'] for b in bins], mode='lines+markers', name='Predicted map bin mean', error_y={'type':'data','array':[b['S_hat_std'] for b in bins],'visible':True}),row=1,col=1)
fig.add_trace(go.Scatter(x=[0,0.8],y=[0,0.8],mode='lines',line={'dash':'dash','color':'gray'},name='Identity reference'),row=1,col=1)
fig.add_trace(go.Bar(x=['SA-13 seed 1','Constant validation mean'],y=[s['mae_model'],s['mae_constant_val_mean']],name='MAE',showlegend=False),row=1,col=2)
fig.update_xaxes(title_text='Ground truth S bin mean',range=[0,0.8],row=1,col=1)
fig.update_yaxes(title_text='Predicted map mean (+/- bin standard deviation)',range=[0,0.8],row=1,col=1)
fig.update_yaxes(title_text='MAE',rangemode='tozero',row=1,col=2)
fig.update_layout(template='plotly_white',width=1250,height=520,title='Estimator diagnostic: SA-13 seed 1; synthetic validation only')
notes = ('Source: '+source.relative_to(root).as_posix()+'; SHA-256='+hashlib.sha256(raw).hexdigest()+'\n'
    'Training checkpoint source: xunanhv/sad-train V2/355813018; SA-13_seed1/final.pth.\n'
    'No new model inference. Error bars are pixel standard deviation within each severity bin, not confidence intervals.\n'
    'Spearman within image R>=2='+str(s['spearman_within_image_R_ge_2'])+'; 30 images.\n'
    'The constant baseline uses ground-truth mean on validation; this is a diagnostic reference, not a trained competitor.\n'
    'No causal conclusion, no synthetic test/GoPro map claim, and no unblinding of SA-11/12.')
html=fig.to_html(full_html=True,include_plotlyjs=True,config={'toImageButtonOptions':{'format':'png','scale':2}})
html=html.replace('</body>','<pre>'+escape(notes)+'</pre></body>')
output=root/'results/eval/estimator_diag_sa13_val40.html'
assert not output.exists(),'Existing chart protected; do not overwrite.'
with output.open('x',encoding='utf-8') as f:
    f.write(html)
print(json.dumps({'plot':str(output),'bytes':output.stat().st_size,'sha256':hashlib.sha256(output.read_bytes()).hexdigest(),'source_sha256':hashlib.sha256(raw).hexdigest(),'images':40,'model_inference':False,'visual_review':'pending'},allow_nan=False))


In [ ]:
from pathlib import Path
import hashlib, json, math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
root = Path.cwd().resolve()
if not (root / 'AGENTS.md').is_file():
    root = root.parent
source = root / 'results/artifacts/kaggle/355813018/original/sad/results/SA-13_seed1/diag.jsonl'
rows = [json.loads(x) for x in source.read_text(encoding='utf-8').splitlines() if x.strip()]
assert len(rows) == 60 and [r['step'] for r in rows] == list(range(500, 30001, 500))
stages = ['enc0','enc1','enc2','enc3','mid0','dec0','dec1','dec2','dec3']
for row in rows:
    for prefix in ('w_sft_', 'dgamma_', 'beta_'):
        assert all(math.isfinite(row[prefix+s]) and row[prefix+s] > 0 for s in stages)
source_sha = hashlib.sha256(source.read_bytes()).hexdigest()
colors = ['#00429d','#4771b2','#73a2c6','#a5d5d8','#b2182b','#e08214','#aa6600','#6a3d9a','#238b45']
fig = make_subplots(rows=1, cols=3, subplot_titles=['Norm lớp cuối SFT head','Mean abs delta gamma','Mean abs bias'])
for col,prefix in enumerate(('w_sft_','dgamma_','beta_'),1):
    for stage,color in zip(stages,colors):
        fig.add_trace(go.Scatter(x=[r['step'] for r in rows],y=[r[prefix+stage] for r in rows],name=stage,mode='lines',legendgroup=stage,showlegend=col==1,line={'color':color,'width':2.4 if stage=='mid0' else 1.6,'dash':'dash' if stage=='mid0' else 'solid'}),row=1,col=col)
    fig.update_yaxes(type='log', title_text='Log scale',row=1,col=col)
    fig.update_xaxes(title_text='Training step',row=1,col=col)
fig.update_layout(template='plotly_white',width=1320,height=600,font={'family':'Arial','size':15},title={'text':'SA-13 seed 1: SFT head diagnostic trên training patches','x':0.05},margin={'t':105,'b':120},legend={'orientation':'h','y':-0.18,'x':0},annotations=list(fig.layout.annotations)+[{'text':'60 diagnostic rows, 500..30.000 step; middle head nhỏ. Không suy ra hiệu quả test hoặc quan hệ nhân quả.','x':0.5,'y':-0.29,'xref':'paper','yref':'paper','showarrow':False,'font':{'size':13}}])
output=root/'results/eval/modulation_training_sa13_seed1.html'
assert not output.exists(), 'Output exists; stop without overwriting.'
html=fig.to_html(full_html=True,include_plotlyjs=True)
html=html.replace('</body>', '<p>Source: '+source.relative_to(root).as_posix()+'; SHA-256 '+source_sha+'; training V2/355813018. Training diagnostics are not test metrics.</p></body>')
with output.open('x',encoding='utf-8') as stream:stream.write(html)
print(json.dumps({'plot':str(output),'bytes':output.stat().st_size,'sha256':hashlib.sha256(output.read_bytes()).hexdigest(),'source_sha256':source_sha,'training_rows':len(rows),'visual_qa':'pending'}))


### Bằng chứng SA-14 numeric diagnostic (V1/357568385)

Diagnostic archive đã VerifiedLocal (CRC/manifest11/11,846JSONLrecords,source/config khớp). Đây là bằng chứng thất bại, không phải checkpoint full30k. Lượt forward35 input vàtham số trướcforward hữu hạn; outputNaN đầu tiên tại encoders.3.18.conv3 sau khi activation/channel trongenc3 tăng rất lớn. CharbonnierNaN,L_map0,0323069; gradient vàoptimizer làm hỏng748parameter tensors, rồiguardabortởcounter84.

Giả thuyết khuếch đại nhánh channel/encoder cần can thiệp kiểm chứng; chưa chứng minh nguyên nhân duy nhất. Thayđổi channel lấyfeature sauLayerNorm vàguardfail-fast mới là đề xuất chưaápdụng, cần quyết định rõ vàpilot riêng. Source: results/artifacts/kaggle/357568385/download-provenance.json; archiveSHA256b62fde4663f21fefebedeb13ac135aab903cd35634e41dcb2e2a28b83199b1df.


### A1/A2 seed2: training và checkpoint đã xác minh local

SavedV1/357522661: SA12seed2/A1 vàSA13seed2/A2 đều30.000bước; checkpointfinite/strict-load/resume/final=last-model/logQC đạt, khônginference smoke/evaluation. Receipt: results/artifacts/kaggle/357522661/download-provenance.json. MỗiGPUwall~9,45giờ; tổngphysicalGPUtraining~18,91giờ, khôngđồngnhấtquota hoặc elapsed phiên.

| Run | Val40 PSNR (monitoring, dB) | Val map MAE | Val map rho | Trạng thái test |
|---|---:|---:|---:|---|
| SA12seed2/A1 | 33.802649 | 0.662274 | -0.005521 | Chưa có CSV |
| SA13seed2/A2 | 33.799719 | 0.176816 | 0.037706 | Chưa có CSV |

Chưa gộp sốval với bảngGoPro, chưa thống kê testnhiềuseed hoặcAccepted. Seed3 vàA3full cònthiếu; Eprobe/oracle/shuffle vẫn cần.
